In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from qwen_vl_utils import process_vision_info
import torch
import re

print("🧠 Initializing Video Analysis AI (Qwen-VL + Custom LoRA)...")

model_path = "Qwen/Qwen2-VL-2B-Instruct"

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_path, dtype=torch.float16, device_map="cuda",
)

adapter_path = "/content/football_checkpoints_REAL"

try:
    video_model = PeftModel.from_pretrained(base_model, adapter_path, is_trainable=False)
    print("✅ Fine-tuned LoRA weights loaded successfully!")
except Exception as e:
    print(f"⚠️ Could not load custom weights. Using base model. Error: {e}")
    video_model = base_model

video_processor = AutoProcessor.from_pretrained(model_path)

In [ ]:
def analyze_video_chunk(chunk_path: str):
    """Sends a short video clip to Qwen and returns the exact action and score."""
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "video", "video": chunk_path,
                    "max_pixels": 360 * 420, "fps": 5.0,
                    "video_reader_backend": "decord"
                },
                {
                    "type": "text",
                    "text": (
                        "Classify the primary football action in this 5-second clip. "
                        "Choose EXACTLY ONE word from this list: pass, score, dribbling, tackling, none.\n"
                        "Rate the execution quality from 1 to 10.\n"
                        "STRICT FORMAT REQUIRED:\n"
                        "Action: [word]\n"
                        "Quality: [X.X]"
                    )
                }
            ]
        }
    ]

    text = video_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = video_processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to("cuda")

    generated_ids = video_model.generate(**inputs, max_new_tokens=64, do_sample=False, temperature=0.0)
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = video_processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]

    action_match = re.search(r"Action:\s*([A-Za-z_]+)", output_text, re.IGNORECASE)
    quality_match = re.search(r"Quality:\s*([\d\.]+)", output_text, re.IGNORECASE)

    action = action_match.group(1).lower() if action_match else "none"
    score = float(quality_match.group(1).rstrip('.')) if quality_match else 0.0

    # Strict guardrail for allowed actions
    allowed_actions = ["pass", "score", "dribbling", "tackling"]
    if action not in allowed_actions:
        action = "none"
        score = 0.0

    return action, score

In [ ]:
import subprocess
import os
import time

def run_live_match_simulation(long_video_path: str, window_size=5, stride=2, total_duration=55):
    player_live_stats = {
        "Passes": 0, "Tackles": 0, "Dribbles": 0, "Shots/Goals (Score)": 0,
        "Saves": 0, "Average Score": 0.0
    }
    score_history = []
    event_log = []
    last_detected_action = None
    cooldown_counter = 0

    print(f"🎥 Starting live analysis on {long_video_path}...")

    for start_time in range(0, total_duration - window_size + 1, stride):
        chunk_filename = f"temp_live_chunk_{start_time}.mp4"
        cmd = f'ffmpeg -y -ss {start_time} -t {window_size} -i "{long_video_path}" -c:v libx264 -preset ultrafast -an -loglevel error "{chunk_filename}"'
        subprocess.run(cmd, shell=True)

        if not os.path.exists(chunk_filename) or os.path.getsize(chunk_filename) < 1000:
            break

        action, score = analyze_video_chunk(chunk_filename)
        timestamp = f"00:{start_time:02d}-00:{start_time+window_size:02d}"

        if score < 2.0: action = "none"
        event_log.append(f"[{timestamp}] {action.upper()} - Quality: {score}")

        if action not in ["none", "unknown"]:
            if action != last_detected_action or cooldown_counter > 1:
                key_map = {
                    "pass": "Passes", "tackling": "Tackles",
                    "dribbling": "Dribbles", "score": "Shots/Goals (Score)", "save": "Saves"
                }
                stat_key = key_map.get(action)

                if stat_key:
                    player_live_stats[stat_key] += 1
                    score_history.append(score)
                    player_live_stats["Average Score"] = round(sum(score_history) / len(score_history), 1)

                last_detected_action = action
                cooldown_counter = 0
            else:
                cooldown_counter += 1
        else:
            cooldown_counter += 1

        print("\033[2J\033[H", end="")
        print(f"🔴 LIVE MATCH FEED | Window: {timestamp}")
        print(f"⚡ CURRENT ACTION: {action.upper()}")
        print("-" * 30)
        for key, value in player_live_stats.items(): print(f"   - {key}: {value}")

        if os.path.exists(chunk_filename): os.remove(chunk_filename)

    return player_live_stats, event_log

In [ ]:
def generate_player_card(player_index, player_df, medical_model, features):
    """Generates static physical stats and predicts health score using Random Forest."""
    player_data = player_df.iloc[player_index]

    pace = int((player_data['Max_Speed'] / player_df['Max_Speed'].max()) * 99)
    stamina = int((player_data['Distance'] / player_df['Distance'].max()) * 99)
    physical = int((player_data['Acceleration_Count'] / player_df['Acceleration_Count'].max()) * 99)

    input_data = player_data[features].values.reshape(1, -1)
    injury_risk_prob = medical_model.predict_proba(input_data)[0][1]
    health_score = 100 - int(injury_risk_prob * 100)

    status = "Fit to Play"
    if health_score < 60: status = "High Injury Risk - Rest Needed"
    elif health_score < 85: status = "Fatigued - Monitor Carefully"

    return {

        "Name": str(player_data['Name']),
        "Position": str(player_data['Position']),
        "FIFA_Stats": {"PAC": pace, "STA": stamina, "PHY": physical},
        "Medical_Report": {
            "Health_Score": health_score,
            "Status": status,
            "Sleep_Quality": float(player_data['Sleep_Quality']),
            "Soreness_Level": float(player_data['Soreness'])
        }
    }

def calculate_market_value(performance_score, health_score, age=24):
    """Calculates a transfer fee range based on AI inputs."""

    base_value = (float(performance_score) ** 2) * 300_000
    health_factor = float(health_score) / 100.0

    if health_score < 60: health_factor *= 0.5
    elif health_score < 80: health_factor *= 0.8

    age_factor = 1.0
    if age < 23: age_factor = 1.3
    if age > 29: age_factor = 0.7

    estimated_value = base_value * health_factor * age_factor
    min_price = round(estimated_value * 0.9, -4)
    max_price = round(estimated_value * 1.1, -4)

    return min_price, max_price

In [ ]:
import plotly.graph_objects as go

def generate_scouting_report_prompt(player_index, video_performance_score, player_df, medical_model, features):
    """Builds the final prompt to feed into the LLM Manager."""
    card = generate_player_card(player_index, player_df, medical_model, features)
    health_score = card['Medical_Report']['Health_Score']

    min_val, max_val = calculate_market_value(video_performance_score, health_score)

    prompt = f"""
    *** CONFIDENTIAL SCOUTING REPORT ***

    TARGET PLAYER: {card['Name']}
    POSITION: {card['Position']}

    --- AI ANALYSIS RESULTS ---
    1. VIDEO ANALYSIS (Technical): {video_performance_score}/10
       (Source: Computer Vision Model)

    2. MEDICAL ANALYSIS (Physical): {health_score}/100
       (Source: Wearable Data Model)
       - Sleep Quality: {card['Medical_Report']['Sleep_Quality']}
       - Recent Soreness: {card['Medical_Report']['Soreness_Level']}

    --- FINANCIAL VALUATION ---
    Recommended Bid Interval: ${min_val:,.0f} - ${max_val:,.0f}

    --- TASK ---
    Act as a Chief Scout. Write a brief recommendation to the Club Director.
    Explain WHY we should (or should not) buy this player based on the data above.
    If the Health Score is low, warn the director about the risk.
    """
    return prompt

def visualize_player_profile(player_index, video_score, player_df, medical_model, features):
    """
    Returns spider diagram data as JSON for the backend to render.
    Does NOT call fig.show() — safe to run outside notebook.
    """
    card_data = generate_player_card(player_index, player_df, medical_model, features)
    normalized_video_score = video_score * 10

    spider_data = {
        "player_name" : card_data['Name'],
        "position"    : card_data['Position'],
        "categories"  : ["Pace", "Stamina", "Physicality", "Health", "Technical"],
        "scores"      : [
            card_data['FIFA_Stats']['PAC'],
            card_data['FIFA_Stats']['STA'],
            card_data['FIFA_Stats']['PHY'],
            card_data['Medical_Report']['Health_Score'],
            normalized_video_score,
        ],
        "medical_report" : card_data['Medical_Report'],
        "fifa_stats"     : card_data['FIFA_Stats'],
    }

    print(f"📊 Spider JSON ready for backend → {card_data['Name']}")
    return spider_data   # ✅ backend renders this, not notebook